In [0]:
# GenerateGridInputs notebook

import json
import math

# tu grid
datasets = ["20news", "csv"]
seeds = [13, 42, 1234]
lrs = [1e-3, 2e-4]
max_lens = [256, 512, 1024]
config_models = [("ffn", 15, 128), ("lstm", 15, 128), ("transformer_scratch", 15, 128)]

EXPERIMENT_PATH = "/Shared/project_test"
DATA_HOME = "/tmp/sklearn_20news"

# === ESTRATEGIA DE PARALELISMO (elige UNA) ===
# GPU_POOLS = ["0,1", "2,3"]
# NUM_PROCESSES = 2
# CONCURRENCY = 2

# GPU_POOLS = ["0", "1", "2", "3", "4", "5", "6", "7"]
GPU_POOLS = ["0", "1", "2", "3"]
NUM_PROCESSES = 1
CONCURRENCY = 4

# -----------------------------
# Batch size formula (with IF)
# -----------------------------
L_REF = 256
BS_MIN = 4

def suggest_batch_size(base_bs_ref: int, max_len: int, model_type: str,
                       L_ref: int = L_REF, bs_min: int = BS_MIN) -> int:
    """
    Returns batch_size_per_gpu computed by:
        bs = floor(base_bs_ref * (L_ref/max_len)^p)
    where:
        p=2 for transformers, p=1 otherwise
    """
    if max_len <= 0:
        return bs_min

    mt = (model_type or "").lower()
    p = 2 if "transformer" in mt else 1

    bs = math.floor(base_bs_ref * (L_ref / max_len) ** p)
    return max(bs_min, bs)

inputs = []
i = 0

for max_len in max_lens:
    for (model_type, epochs, base_bs_ref) in config_models:
        bs = suggest_batch_size(base_bs_ref, max_len, model_type)

        for seed in seeds:
            for lr in lrs:
                for dataset in datasets:
                    inputs.append({
                        "dataset": dataset,
                        "model_type": model_type,
                        "seed": seed,
                        "lr": lr,
                        "max_len": max_len,
                        "base_batch_size": bs,
                        "epochs": epochs,
                        "num_processes": NUM_PROCESSES,
                        "local_mode": "true",
                        "experiment_path": EXPERIMENT_PATH,
                        "enable_mlflow": "true",
                        "data_home": DATA_HOME,
                        "gpu_ids": GPU_POOLS[i % len(GPU_POOLS)],
                    })
                    i += 1

print("Total runs:", len(inputs))
print(json.dumps(inputs, indent=2))

# 👇 esto es lo clave: guardar el array como task value para que el For each lo consuma
dbutils.jobs.taskValues.set(key="grid", value=inputs)

# opcional: también guarda la concurrencia para usarla “a mano” en la UI
dbutils.jobs.taskValues.set(key="concurrency", value=CONCURRENCY)

Total runs: 108
[
  {
    "dataset": "20news",
    "model_type": "ffn",
    "seed": 13,
    "lr": 0.001,
    "max_len": 256,
    "base_batch_size": 128,
    "epochs": 10,
    "num_processes": 1,
    "local_mode": "true",
    "experiment_path": "/Shared/Comp_20news_V1",
    "enable_mlflow": "true",
    "data_home": "/tmp/sklearn_20news",
    "gpu_ids": "0"
  },
  {
    "dataset": "csv",
    "model_type": "ffn",
    "seed": 13,
    "lr": 0.001,
    "max_len": 256,
    "base_batch_size": 128,
    "epochs": 10,
    "num_processes": 1,
    "local_mode": "true",
    "experiment_path": "/Shared/Comp_20news_V1",
    "enable_mlflow": "true",
    "data_home": "/tmp/sklearn_20news",
    "gpu_ids": "1"
  },
  {
    "dataset": "20news",
    "model_type": "ffn",
    "seed": 13,
    "lr": 0.0001,
    "max_len": 256,
    "base_batch_size": 128,
    "epochs": 10,
    "num_processes": 1,
    "local_mode": "true",
    "experiment_path": "/Shared/Comp_20news_V1",
    "enable_mlflow": "true",
    "data_hom